# Capstone build --- Chapter 15: The Assembled Capstone

Each chapter of this series added one layer: the typed action (Chapter~1), the loop (Chapter~1), the reasoning trace (Chapter~3), the task and action types (Chapter~4), the five-tool action space (Chapter~5), the gate stack (Chapter~6), the budget (Chapter~7), the plan (Chapter~8), the working memory (Chapter~9), trajectory evaluation (Chapter~10), the designed suite (Chapter~11), runtime governance (Chapter~12) and escalation (Chapter~13). `build_complaint_harness` assembles them. This chapter checks that the assembled object is exactly the one the layers describe, then runs it.

## Assemble the harness

One call assembles the whole capstone: it registers the five tools, builds the policy engine, loads the trained plausibility gate and wraps the executor in the governance harness. The return is the `(harness, registry)` the shipped agent runs behind.

In [ ]:
import json
from pathlib import Path
from agentlab.capstone import build_complaint_harness
from agentlab.core import Budget, BudgetTracker, TaskSpec

root = next((c for c in (Path('.'), Path('..'), Path('../code'), Path('code'))
             if (c / 'data' / 'eval_cases' / 'cases.json').exists()), Path('.'))
cases = json.loads((root / 'data' / 'eval_cases' / 'cases.json').read_text())
harness, registry = build_complaint_harness(policies_dir=root / 'data' / 'policies')

## The wiring matches what the series built

The claim of the series is that the layers add up to the shipped harness. That is checkable at the level of the wiring: the action space is exactly the five typed tools of Chapter~5, and the gate stack is exactly the three gates of Chapters~6 and~12 --- syntax, policy, plausibility --- with the trained gate last. The assertions below fail if the assembled harness and the described layers ever drift apart.

In [ ]:
from agentlab.governance import SyntaxGate
from agentlab.gms_backend import GMSPlausibilityGate

tool_names = {t.name for t in registry.all()}
expected_tools = {'classify_complaint', 'extract_facts', 'search_policy',
                  'flag_regulatory', 'draft_response'}
assert tool_names == expected_tools, tool_names
print('action space :', sorted(tool_names))

gates = harness._executor._gates
print('gate stack   :', [g.__class__.__name__ for g in gates])
assert len(gates) == 3
assert isinstance(gates[0], SyntaxGate)
assert isinstance(gates[-1], GMSPlausibilityGate)
print('wiring matches the shipped harness: OK')

## A routine case is handled

A well-formed complaint runs the full workflow to a drafted reply: classify, extract, search policy, flag regulatory, draft. The final output carries the classification, the governing policy evidence and the draft.

In [ ]:
case = next(c for c in cases if c['id'] == 'case-002')
task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
out = traj.final_state.final_output or {}
print('message       :', case['message'])
print('status        :', traj.final_state.status)
print('classification :', out.get('classification'))
print('action        :', out.get('recommended_action'))
print('draft         :', (out.get('draft_response') or '')[:120])

## An adversarial case escalates

A case that carries regulatory risk does not get a drafted reply: the agent escalates, and the audit chain records the decision. Verifying the chain confirms the run was not altered after the fact.

In [ ]:
case = next(c for c in cases if c['id'] == 'case-016')
task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
esc = next((r for r in traj.records if r.action.kind == 'escalate'), None)
print('message   :', case['message'])
print('status    :', traj.final_state.status)
print('escalation:', esc.action.reason if esc else '(none)')
print('audit chain valid:', harness.audit.verify())
assert harness.audit.verify()

The capstone is the sum of the layers: a fixed workflow of five typed tools, run behind a three-gate stack, recorded in a verifiable audit chain, escalating when it reaches the edge of its authority. What this series built by hand is the architecture; what it imported is the trained geometry and the language models inside the tools. The object assembled here is the one `build_complaint_harness` ships, and Chapter~16 tests it against the designed suite of Chapter~11.